# Seq2Seq LSTM - English to Vietnamese Translation Training

Self-contained notebook with full LSTM Seq2Seq implementation.

## Pipeline
1. Install dependencies
2. Define Seq2Seq LSTM model (all classes inline)
3. Download & preprocess OPUS-100 En-Vi data
4. Train SentencePiece tokenizer
5. PyTorch Dataset / DataLoader
6. Training loop
7. Greedy decoding inference
8. BLEU evaluation
9. Save checkpoint

**Author:** Khoa


---
## 1. Setup & Imports

In [ ]:
import math, os, sys, random, time, urllib.request, shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 42
torch.manual_seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

---
## 2. Hyperparameters

In [ ]:
VOCAB_SIZE = 16000
PAD_IDX = 0; SOS_IDX = 1; EOS_IDX = 2; UNK_IDX = 3
MAX_SEQ_LEN = 128

EMB_DIM = 256; HID_DIM = 512; N_LAYERS = 2; DROPOUT = 0.5

BATCH_SIZE = 64; NUM_EPOCHS = 30
LEARNING_RATE = 1e-3; CLIP = 1.0

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', '..', 'data', 'processed')
CHECKPOINT_DIR = os.path.join(NOTEBOOK_DIR, 'checkpoints')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Data: {DATA_DIR}')
print(f'Checkpoints: {CHECKPOINT_DIR}')

---
## 3. Seq2Seq LSTM Model (full implementation)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: (seq_len, batch)
        embedded = self.dropout(self.embedding(src))
        _, (hidden, cell) = self.rnn(embedded)
        # hidden, cell: (n_layers, batch, hid_dim)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inp, hidden, cell):
        # inp: (batch,)
        inp = inp.unsqueeze(0)  # (1, batch)
        embedded = self.dropout(self.embedding(inp))
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        # output: (1, batch, hid_dim)
        prediction = self.fc_out(output.squeeze(0))
        # prediction: (batch, output_dim)
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (src_len, batch)
        # trg: (trg_len, batch)
        hidden, cell = self.encoder(src)

        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)

        inp = trg[0, :]  # <sos> token

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(inp, hidden, cell)
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            inp = trg[t] if teacher_force else output.argmax(1)

        return outputs

---
## 4. SentencePiece Tokenizer

In [ ]:
try:
    import sentencepiece as spm
except ImportError:
    !pip install sentencepiece -q
    import sentencepiece as spm

TOKENIZER_MODEL = os.path.join(DATA_DIR, 'sp_envi.model')

def train_tokenizer(src_file, tgt_file, model_prefix, vocab_size=16000):
    combined = model_prefix + '_combined.txt'
    with open(combined, 'w', encoding='utf-8') as out:
        for f in [src_file, tgt_file]:
            with open(f, 'r', encoding='utf-8') as fh:
                for line in fh:
                    out.write(line.strip() + '\n')
    spm.SentencePieceTrainer.train(
        input=combined, model_prefix=model_prefix,
        vocab_size=vocab_size, character_coverage=1.0,
        model_type='bpe',
        pad_id=PAD_IDX, unk_id=UNK_IDX, bos_id=SOS_IDX, eos_id=EOS_IDX
    )
    os.remove(combined)
    print(f'Tokenizer saved to {model_prefix}.model')

if os.path.exists(TOKENIZER_MODEL):
    sp = spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL)
    print(f'Loaded tokenizer. Vocab size: {sp.get_piece_size()}')
else:
    print('Tokenizer not found. It will be trained after data download.')

---
## 5. Download OPUS-100 En-Vi Data

In [ ]:
EN_TRAIN = os.path.join(DATA_DIR, 'train.en')
VI_TRAIN = os.path.join(DATA_DIR, 'train.vi')
EN_VAL = os.path.join(DATA_DIR, 'val.en')
VI_VAL = os.path.join(DATA_DIR, 'val.vi')

if not os.path.exists(EN_TRAIN):
    base = 'https://object.pouta.csc.fi/OPUS-100/v1.0/raw/'
    for local, name in [(EN_TRAIN, 'train.en'), (VI_TRAIN, 'train.vi'),
                        (EN_VAL, 'dev.en'), (VI_VAL, 'dev.vi')]:
        url = base + 'en-vi/' + name
        print(f'Downloading {name}...')
        urllib.request.urlretrieve(url, local)
    print('Download complete.')
else:
    print('Data already exists.')

In [ ]:
if not os.path.exists(TOKENIZER_MODEL):
    train_tokenizer(EN_TRAIN, VI_TRAIN, TOKENIZER_MODEL.replace('.model', ''), VOCAB_SIZE)
    sp = spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL)

print(f'Vocab size: {sp.get_piece_size()}')
print('Sample:', sp.encode('Hello, how are you?', out_type=str))

---
## 6. PyTorch Dataset & DataLoader

Note: LSTM expects `(seq_len, batch)` format (not batch-first). The collate function handles the transpose.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, src_path, tgt_path, sp, max_len=128, max_samples=None):
        self.src, self.tgt = [], []
        self.sp = sp; self.max_len = max_len
        with open(src_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.src.append(line)
        with open(tgt_path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.tgt.append(line)
        if max_samples:
            self.src = self.src[:max_samples]
            self.tgt = self.tgt[:max_samples]
        assert len(self.src) == len(self.tgt)

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src = self.sp.encode(self.src[idx], out_type=int)[:self.max_len-2]
        tgt = self.sp.encode(self.tgt[idx], out_type=int)[:self.max_len-2]
        return (torch.tensor([SOS_IDX] + src + [EOS_IDX], dtype=torch.long),
                torch.tensor([SOS_IDX] + tgt + [EOS_IDX], dtype=torch.long))

def collate_fn(batch):
    src, tgt = zip(*batch)
    # LSTM expects (seq_len, batch) -> transpose from batch_first
    src = pad_sequence(src, batch_first=False, padding_value=PAD_IDX)
    tgt = pad_sequence(tgt, batch_first=False, padding_value=PAD_IDX)
    return src, tgt

MAX_SAMPLES = None  # set to e.g. 5000 for a quick test

train_dataset = TranslationDataset(EN_TRAIN, VI_TRAIN, sp, MAX_SEQ_LEN, MAX_SAMPLES)
val_dataset = TranslationDataset(EN_VAL, VI_VAL, sp, MAX_SEQ_LEN, MAX_SAMPLES)

train_loader = DataLoader(train_dataset, BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

---
## 7. Model, Optimizer & Loss

In [ ]:
actual_vocab_size = sp.get_piece_size()
print(f'Vocab size: {actual_vocab_size}')

encoder = Encoder(actual_vocab_size, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
decoder = Decoder(actual_vocab_size, EMB_DIM, HID_DIM, N_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, device).to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
print('Ready.')

---
## 8. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip):
    model.train()
    total = 0
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        # src: (src_len, batch), trg: (trg_len, batch)
        optimizer.zero_grad()
        output = model(src, trg)  # teacher forcing = 0.5
        # output: (trg_len, batch, vocab)
        output = output[1:].reshape(-1, actual_vocab_size)
        trg = trg[1:].reshape(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total = 0
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, 0)  # no teacher forcing
            output = output[1:].reshape(-1, actual_vocab_size)
            trg = trg[1:].reshape(-1)
            total += criterion(output, trg).item()
    return total / len(loader)

print('Functions defined.')

In [ ]:
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP)
    val_loss = validate(model, val_loader, criterion)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
          f'Time: {time.time()-t0:.1f}s')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss,
        }, os.path.join(CHECKPOINT_DIR, 'lstm_best.pt'))
        print(f'  --> Saved best ({val_loss:.4f})')

print('Training complete.')

---
## 9. Greedy Decoding (Inference)

In [ ]:
def translate(model, src_sentence, sp, max_len=128):
    model.eval()
    src_ids = [SOS_IDX] + sp.encode(src_sentence, out_type=int) + [EOS_IDX]
    src = torch.tensor(src_ids, dtype=torch.long).unsqueeze(1).to(device)  # (seq, 1)

    with torch.no_grad():
        hidden, cell = model.encoder(src)
        inp = torch.tensor([SOS_IDX], device=device)  # (1,)
        tokens = []
        for _ in range(max_len):
            output, hidden, cell = model.decoder(inp, hidden, cell)
            top1 = output.argmax(-1).item()
            if top1 == EOS_IDX:
                break
            tokens.append(top1)
            inp = torch.tensor([top1], device=device)

    return sp.decode(tokens)

print('Ready.')

In [ ]:
for s in ['Hello, how are you?', 'The weather is nice today.',
          'I would like to learn more about machine learning.']:
    print(f'EN: {s}\nVI: {translate(model, s, sp)}\n')

---
## 10. BLEU Evaluation

In [ ]:
try:
    import sacrebleu
except ImportError:
    !pip install sacrebleu -q
    import sacrebleu

def compute_bleu(model, val_dataset, sp, n=200):
    model.eval()
    refs, hyps = [], []
    for idx in random.sample(range(len(val_dataset)), min(n, len(val_dataset))):
        refs.append(val_dataset.tgt[idx])
        hyps.append(translate(model, val_dataset.src[idx], sp))
    return sacrebleu.corpus_bleu(hyps, [refs]).score

bleu = compute_bleu(model, val_dataset, sp, 200)
print(f'Validation BLEU: {bleu:.2f}')

---
## 11. Save Final Model

In [ ]:
final_path = os.path.join(CHECKPOINT_DIR, 'lstm_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'model_config': {
        'emb_dim': EMB_DIM, 'hid_dim': HID_DIM,
        'n_layers': N_LAYERS, 'dropout': DROPOUT,
    },
    'vocab_size': actual_vocab_size,
    'history': history, 'bleu': bleu,
}, final_path)
shutil.copy(TOKENIZER_MODEL, os.path.join(CHECKPOINT_DIR, 'sp_envi.model'))
print(f'Saved to {final_path}')

---
## 12. Load Checkpoint & Resume (optional)

In [ ]:
def load_checkpoint(path, model, optimizer=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f'Loaded from {path}')
    return ckpt.get('epoch', 0), ckpt.get('val_loss', float('inf'))

# epoch, loss = load_checkpoint(CHECKPOINT_DIR + '/lstm_best.pt', model, optimizer)